# Actividad 3 - Aprendizaje supervisado y no supervisado con PySpark

**Materia:** TC4034.10 - Análisis de grandes volúmenes de datos  
**Dataset:** Instacart Market Basket Analysis  
**Nombre:** Roberto Augusto Olivares González  
**Matrícula:** A01796223  

---

## Objetivo de la actividad

Aplicar algoritmos de aprendizaje supervisado y no supervisado mediante PySpark para resolver un problema de análisis de datos sobre grandes volúmenes de información. Para ello se construye una muestra representativa **M** a partir del dataset Instacart Market Basket Analysis, se preparan los datos, se divide la muestra en conjuntos de entrenamiento y prueba, y se entrenan dos modelos: uno supervisado y uno no supervisado.

# 1. Introducción

El aprendizaje automático puede dividirse, de forma general, en aprendizaje supervisado y aprendizaje no supervisado.

El **aprendizaje supervisado** utiliza datos históricos que contienen una variable objetivo conocida. A partir de ejemplos etiquetados, el algoritmo aprende una relación entre las variables predictoras y la variable objetivo. Este enfoque se utiliza en problemas de clasificación y regresión. En PySpark, MLlib incluye algoritmos supervisados como Decision Tree, Random Forest, Gradient-Boosted Trees, Logistic Regression, Linear Regression y Multilayer Perceptron.

El **aprendizaje no supervisado** se aplica cuando no existe una variable objetivo previamente definida. El propósito es descubrir estructuras, patrones o grupos dentro de los datos. En PySpark, MLlib incluye algoritmos no supervisados como KMeans, Bisecting KMeans, Gaussian Mixture, LDA y Power Iteration Clustering.

Para esta actividad se seleccionan los siguientes algoritmos:

- **RandomForestClassifier** como algoritmo supervisado, debido a que permite resolver problemas de clasificación binaria y manejar relaciones no lineales entre variables.
- **KMeans** como algoritmo no supervisado, debido a que permite agrupar eventos de compra con características similares.

En el caso supervisado, la variable objetivo será `reordered`, la cual indica si un producto fue recomprado (`1`) o no (`0`). En el caso no supervisado, se utilizarán variables relacionadas con el horario, frecuencia de recompra, categoría del producto y comportamiento dentro del carrito para identificar grupos de compras similares.

In [2]:
import os
import sys


java_home_local = "/opt/anaconda3/envs/env_pyspark/lib/jvm"
if os.path.exists(java_home_local):
    os.environ["JAVA_HOME"] = java_home_local
    os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

try:
    import findspark
    findspark.init()
    print("findspark inicializado:", findspark.find())
except Exception as e:
    print("findspark no se inicializó. Si PySpark funciona, este mensaje puede ignorarse.")
    print("Detalle:", e)

findspark inicializado: /opt/anaconda3/envs/env_pyspark/lib/python3.12/site-packages/pyspark


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, lit, when, round as spark_round, concat_ws, rand,
    row_number, monotonically_increasing_id, avg, desc, isnan
)
from pyspark.sql.window import Window

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier, DecisionTreeClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator,
    BinaryClassificationEvaluator,
    ClusteringEvaluator
)

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Actividad3_Instacart_ML_PySpark") \
    .getOrCreate()

spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.sparkContext.setLogLevel("WARN")

print("Spark iniciado correctamente")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/31 07:09:36 WARN Utils: Your hostname, MacBook-Air-de-Roberto-3.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.115 instead (on interface en0)
26/05/31 07:09:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/31 07:09:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark iniciado correctamente


# 3. Selección de los datos

En esta etapa se construye la muestra inicial **M** a partir del dataset Instacart Market Basket Analysis. Se parte de los archivos CSV usados en la actividad previa:

- `orders.csv`
- `order_products__train.csv`
- `products.csv`
- `aisles.csv`
- `departments.csv`

La unidad de análisis corresponde al registro **orden-producto**, es decir, cada producto comprado dentro de una orden. Para evitar tiempos de procesamiento altos, se recupera un número limitado de registros por cada partición previamente definida.

Las particiones se construyen a partir de tres variables derivadas:

- `bloque_horario`, derivada de `order_hour_of_day`.
- `frecuencia_recompra`, derivada de `days_since_prior_order`.
- `grupo_departamento`, derivada de `department`.

La muestra **M** se obtiene mediante muestreo estratificado por partición, tomando hasta 500 registros por cada combinación de las variables anteriores.

In [4]:
archivos_requeridos = [
    "orders.csv",
    "order_products__train.csv",
    "products.csv",
    "aisles.csv",
    "departments.csv"
]

# Ajusta esta ruta si tus archivos están en otra carpeta.
# El código también intenta detectar rutas comunes automáticamente.
rutas_candidatas = [
    "/Users/robertoolivares/Downloads/archive",
    "/content/drive/MyDrive/TC4034_Equipo57/instacart",
    "/content/drive/MyDrive/instacart",
    "/mnt/data"
]

ruta_base = None
for ruta in rutas_candidatas:
    if os.path.exists(ruta) and all(os.path.exists(os.path.join(ruta, archivo)) for archivo in archivos_requeridos):
        ruta_base = ruta
        break

if ruta_base is None:
    raise FileNotFoundError(
        "No se encontró una carpeta con todos los archivos requeridos. "
        "Actualiza la variable rutas_candidatas o asigna manualmente ruta_base."
    )

print("Ruta base seleccionada:", ruta_base)
print("Archivos encontrados:")
for archivo in archivos_requeridos:
    print("-", archivo)

Ruta base seleccionada: /Users/robertoolivares/Downloads/archive
Archivos encontrados:
- orders.csv
- order_products__train.csv
- products.csv
- aisles.csv
- departments.csv


In [5]:
orders = spark.read.csv(f"{ruta_base}/orders.csv", header=True, inferSchema=True)
order_products_train = spark.read.csv(f"{ruta_base}/order_products__train.csv", header=True, inferSchema=True)
products = spark.read.csv(f"{ruta_base}/products.csv", header=True, inferSchema=True)
aisles = spark.read.csv(f"{ruta_base}/aisles.csv", header=True, inferSchema=True)
departments = spark.read.csv(f"{ruta_base}/departments.csv", header=True, inferSchema=True)

print("Datos cargados correctamente")
print("orders:", orders.count())
print("order_products__train:", order_products_train.count())
print("products:", products.count())
print("aisles:", aisles.count())
print("departments:", departments.count())

Datos cargados correctamente
orders: 3421083
order_products__train: 1384617
products: 49688
aisles: 134
departments: 21


In [6]:
orders_train = orders.filter(col("eval_set") == "train")

base = order_products_train \
    .join(orders_train, on="order_id", how="inner") \
    .join(products, on="product_id", how="inner") \
    .join(aisles, on="aisle_id", how="inner") \
    .join(departments, on="department_id", how="inner")

base.cache()

total_base = base.count()
print("Total de registros orden-producto en la base analítica:", total_base)

base.select(
    "order_id", "user_id", "product_id", "product_name",
    "department", "aisle", "order_dow", "order_hour_of_day",
    "days_since_prior_order", "reordered", "add_to_cart_order"
).show(10, truncate=False)

[Stage 33:=================================================>    (185 + 8) / 200]

Total de registros orden-producto en la base analítica: 1384617
+--------+-------+----------+----------------------------------------------------+------------+----------------------+---------+-----------------+----------------------+---------+-----------------+
|order_id|user_id|product_id|product_name                                        |department  |aisle                 |order_dow|order_hour_of_day|days_since_prior_order|reordered|add_to_cart_order|
+--------+-------+----------+----------------------------------------------------+------------+----------------------+---------+-----------------+----------------------+---------+-----------------+
|1342    |156818 |13176     |Bag of Organic Bananas                              |produce     |fresh fruits          |3        |8                |30.0                  |1        |1                |
|1342    |156818 |30827     |Seedless Cucumbers                                  |produce     |packaged produce      |3        |8               

## 3.1 Construcción de variables de particionamiento

Se crean tres variables categóricas para representar dimensiones relevantes del comportamiento de compra:

- **bloque_horario:** agrupa la hora de compra en madrugada, mañana, tarde y noche.
- **frecuencia_recompra:** agrupa los días desde la orden anterior en recompra reciente, media y tardía.
- **grupo_departamento:** agrupa los departamentos en frescos, básicos, consumo recurrente y otros.

In [7]:
base_particionada = base \
    .filter(col("days_since_prior_order").isNotNull()) \
    .withColumn(
        "bloque_horario",
        when((col("order_hour_of_day") >= 0) & (col("order_hour_of_day") <= 5), "madrugada 0-5")
        .when((col("order_hour_of_day") >= 6) & (col("order_hour_of_day") <= 11), "mañana 6-11")
        .when((col("order_hour_of_day") >= 12) & (col("order_hour_of_day") <= 17), "tarde 12-17")
        .when((col("order_hour_of_day") >= 18) & (col("order_hour_of_day") <= 23), "noche 18-23")
        .otherwise("sin clasificar")
    ) \
    .withColumn(
        "frecuencia_recompra",
        when((col("days_since_prior_order") >= 0) & (col("days_since_prior_order") <= 7), "reciente 0-7 días")
        .when((col("days_since_prior_order") >= 8) & (col("days_since_prior_order") <= 14), "media 8-14 días")
        .when((col("days_since_prior_order") >= 15) & (col("days_since_prior_order") <= 30), "tardía 15-30 días")
        .otherwise("sin clasificar")
    ) \
    .withColumn(
        "grupo_departamento",
        when(col("department") == "produce", "frescos")
        .when(col("department") == "dairy eggs", "básicos")
        .when(col("department").isin("snacks", "beverages", "frozen", "pantry"), "consumo recurrente")
        .otherwise("otros")
    ) \
    .withColumn(
        "clave_particion",
        concat_ws(" | ", col("bloque_horario"), col("frecuencia_recompra"), col("grupo_departamento"))
    )

base_particionada.cache()

total_particionada = base_particionada.count()
print("Total de registros disponibles para particionamiento:", total_particionada)

base_particionada.select(
    "order_id", "product_name", "department", "order_hour_of_day",
    "days_since_prior_order", "bloque_horario", "frecuencia_recompra", "grupo_departamento",
    "clave_particion"
).show(10, truncate=False)

[Stage 42:==========================================>           (158 + 9) / 200]

Total de registros disponibles para particionamiento: 1384617
+--------+----------------------------------------------------+------------+-----------------+----------------------+--------------+-------------------+------------------+----------------------------------------------------+
|order_id|product_name                                        |department  |order_hour_of_day|days_since_prior_order|bloque_horario|frecuencia_recompra|grupo_departamento|clave_particion                                     |
+--------+----------------------------------------------------+------------+-----------------+----------------------+--------------+-------------------+------------------+----------------------------------------------------+
|1342    |Bag of Organic Bananas                              |produce     |8                |30.0                  |mañana 6-11   |tardía 15-30 días  |frescos           |mañana 6-11 | tardía 15-30 días | frescos           |
|1342    |Seedless Cucumbers          

In [8]:
particiones = base_particionada \
    .groupBy("bloque_horario", "frecuencia_recompra", "grupo_departamento", "clave_particion") \
    .agg(count("*").alias("total_registros")) \
    .withColumn("probabilidad_ocurrencia", col("total_registros") / lit(total_particionada)) \
    .withColumn("porcentaje_ocurrencia", spark_round(col("probabilidad_ocurrencia") * 100, 4)) \
    .orderBy("bloque_horario", "frecuencia_recompra", "grupo_departamento")

print("Número de particiones encontradas:", particiones.count())
particiones.show(60, truncate=False)

Número de particiones encontradas: 48


[Stage 60:=================================================>    (183 + 8) / 200]

+--------------+-------------------+------------------+------------------------------------------------------+---------------+-----------------------+---------------------+
|bloque_horario|frecuencia_recompra|grupo_departamento|clave_particion                                       |total_registros|probabilidad_ocurrencia|porcentaje_ocurrencia|
+--------------+-------------------+------------------+------------------------------------------------------+---------------+-----------------------+---------------------+
|madrugada 0-5 |media 8-14 días    |básicos           |madrugada 0-5 | media 8-14 días | básicos             |903            |6.521659058064432E-4   |0.0652               |
|madrugada 0-5 |media 8-14 días    |consumo recurrente|madrugada 0-5 | media 8-14 días | consumo recurrente  |1883           |0.0013599428578444435  |0.136                |
|madrugada 0-5 |media 8-14 días    |frescos           |madrugada 0-5 | media 8-14 días | frescos             |1737           |0.0012544

## 3.2 Construcción de la muestra M

Para mantener una muestra de tamaño contenido, se toma un máximo de 500 registros por partición. Con 48 particiones, el tamaño máximo esperado de la muestra es aproximadamente 24,000 registros. Esta estrategia permite conservar representación de todos los grupos definidos, evitando que las particiones con mayor volumen dominen por completo la muestra.

In [9]:
seed = 57
n_por_particion = 500

ventana_particion = Window.partitionBy("clave_particion").orderBy(rand(seed))

muestra_M = base_particionada \
    .withColumn("rn", row_number().over(ventana_particion)) \
    .filter(col("rn") <= n_por_particion) \
    .drop("rn") \
    .cache()

total_M = muestra_M.count()
print("Total de registros en la muestra M:", total_M)

muestra_M.groupBy("clave_particion") \
    .agg(count("*").alias("registros_muestra")) \
    .orderBy("clave_particion") \
    .show(60, truncate=False)

Total de registros en la muestra M: 24000
+------------------------------------------------------+-----------------+
|clave_particion                                       |registros_muestra|
+------------------------------------------------------+-----------------+
|madrugada 0-5 | media 8-14 días | básicos             |500              |
|madrugada 0-5 | media 8-14 días | consumo recurrente  |500              |
|madrugada 0-5 | media 8-14 días | frescos             |500              |
|madrugada 0-5 | media 8-14 días | otros               |500              |
|madrugada 0-5 | reciente 0-7 días | básicos           |500              |
|madrugada 0-5 | reciente 0-7 días | consumo recurrente|500              |
|madrugada 0-5 | reciente 0-7 días | frescos           |500              |
|madrugada 0-5 | reciente 0-7 días | otros             |500              |
|madrugada 0-5 | tardía 15-30 días | básicos           |500              |
|madrugada 0-5 | tardía 15-30 días | consumo recurrente|50

# 4. Preparación de los datos

En esta etapa se corrigen inconsistencias y se prepara la muestra **M** para el entrenamiento de modelos. Las acciones principales son:

1. Selección de columnas relevantes.
2. Eliminación de registros con valores nulos en variables críticas.
3. Conversión de tipos de datos numéricos.
4. Filtrado de rangos válidos para reducir posibles valores atípicos.
5. Creación de la columna `label` para el modelo supervisado.

En este caso, se consideran como rangos válidos:

- `order_dow` entre 0 y 6.
- `order_hour_of_day` entre 0 y 23.
- `days_since_prior_order` entre 0 y 30.
- `add_to_cart_order` mayor o igual a 1.
- `reordered` con valores 0 o 1.

In [10]:
columnas_modelo = [
    "order_id", "user_id", "product_id", "product_name",
    "department", "aisle", "order_dow", "order_hour_of_day",
    "days_since_prior_order", "add_to_cart_order", "reordered",
    "bloque_horario", "frecuencia_recompra", "grupo_departamento", "clave_particion"
]

muestra_preparacion = muestra_M.select(*columnas_modelo)

print("Registros antes de limpieza:", muestra_preparacion.count())

# Revisión de valores nulos por columna.
conteo_nulos = muestra_preparacion.select([
    count(when(col(c).isNull(), c)).alias(c) for c in columnas_modelo
])

conteo_nulos.show(truncate=False)

Registros antes de limpieza: 24000
+--------+-------+----------+------------+----------+-----+---------+-----------------+----------------------+-----------------+---------+--------------+-------------------+------------------+---------------+
|order_id|user_id|product_id|product_name|department|aisle|order_dow|order_hour_of_day|days_since_prior_order|add_to_cart_order|reordered|bloque_horario|frecuencia_recompra|grupo_departamento|clave_particion|
+--------+-------+----------+------------+----------+-----+---------+-----------------+----------------------+-----------------+---------+--------------+-------------------+------------------+---------------+
|0       |0      |0         |0           |0         |0    |0        |0                |0                     |0                |0        |0             |0                  |0                 |0              |
+--------+-------+----------+------------+----------+-----+---------+-----------------+----------------------+-----------------+-

In [11]:
columnas_numericas = [
    "order_dow",
    "order_hour_of_day",
    "days_since_prior_order",
    "add_to_cart_order"
]

columnas_categoricas = [
    "department",
    "aisle",
    "bloque_horario",
    "frecuencia_recompra",
    "grupo_departamento"
]

muestra_limpia = muestra_preparacion \
    .dropna(subset=columnas_numericas + columnas_categoricas + ["reordered"])

for c in columnas_numericas:
    muestra_limpia = muestra_limpia.withColumn(c, col(c).cast("double"))

muestra_limpia = muestra_limpia.withColumn("label", col("reordered").cast("double"))

muestra_limpia = muestra_limpia.filter(
    (col("order_dow") >= 0) & (col("order_dow") <= 6) &
    (col("order_hour_of_day") >= 0) & (col("order_hour_of_day") <= 23) &
    (col("days_since_prior_order") >= 0) & (col("days_since_prior_order") <= 30) &
    (col("add_to_cart_order") >= 1) &
    (col("label").isin(0.0, 1.0))
)

muestra_limpia = muestra_limpia.withColumn("row_id", monotonically_increasing_id()).cache()

print("Registros después de limpieza:", muestra_limpia.count())
muestra_limpia.printSchema()

Registros después de limpieza: 24000
root
 |-- order_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- aisle: string (nullable = true)
 |-- order_dow: double (nullable = true)
 |-- order_hour_of_day: double (nullable = true)
 |-- days_since_prior_order: double (nullable = true)
 |-- add_to_cart_order: double (nullable = true)
 |-- reordered: integer (nullable = true)
 |-- bloque_horario: string (nullable = false)
 |-- frecuencia_recompra: string (nullable = false)
 |-- grupo_departamento: string (nullable = false)
 |-- clave_particion: string (nullable = false)
 |-- label: double (nullable = true)
 |-- row_id: long (nullable = false)



In [12]:
print("Distribución de la variable objetivo reordered / label:")
total_limpia = muestra_limpia.count()

muestra_limpia.groupBy("label") \
    .agg(count("*").alias("registros")) \
    .withColumn("porcentaje", spark_round(col("registros") / lit(total_limpia) * 100, 2)) \
    .orderBy("label") \
    .show()

Distribución de la variable objetivo reordered / label:
+-----+---------+----------+
|label|registros|porcentaje|
+-----+---------+----------+
|  0.0|     9056|     37.73|
|  1.0|    14944|     62.27|
+-----+---------+----------+



# 5. Preparación del conjunto de entrenamiento y prueba

Para el modelo supervisado, la muestra limpia se divide en conjunto de entrenamiento y prueba. Se utiliza una división **80% entrenamiento / 20% prueba**, ya que permite reservar la mayor parte de los datos para el aprendizaje del modelo y conservar un subconjunto independiente para evaluación.

Para reducir el riesgo de sesgo, la división se realiza de forma estratificada considerando la combinación entre la partición original (`clave_particion`) y la variable objetivo (`reordered`). De esta manera, se busca mantener representación de los grupos de comportamiento y de las clases 0 y 1 en ambos conjuntos.

In [13]:
train_fraction = 0.80

muestra_split = muestra_limpia.withColumn(
    "estrato_split",
    concat_ws(" | ", col("clave_particion"), col("label").cast("string"))
)

estratos = [row["estrato_split"] for row in muestra_split.select("estrato_split").distinct().collect()]
fractions = {estrato: train_fraction for estrato in estratos}

train_df = muestra_split.sampleBy("estrato_split", fractions=fractions, seed=seed).cache()
train_ids = train_df.select("row_id")
test_df = muestra_split.join(train_ids, on="row_id", how="left_anti").cache()

print("Registros entrenamiento:", train_df.count())
print("Registros prueba:", test_df.count())

print("Distribución de label en entrenamiento:")
train_total = train_df.count()
train_df.groupBy("label") \
    .agg(count("*").alias("registros")) \
    .withColumn("porcentaje", spark_round(col("registros") / lit(train_total) * 100, 2)) \
    .orderBy("label") \
    .show()

print("Distribución de label en prueba:")
test_total = test_df.count()
test_df.groupBy("label") \
    .agg(count("*").alias("registros")) \
    .withColumn("porcentaje", spark_round(col("registros") / lit(test_total) * 100, 2)) \
    .orderBy("label") \
    .show()

Registros entrenamiento: 19218
Registros prueba: 4782
Distribución de label en entrenamiento:
+-----+---------+----------+
|label|registros|porcentaje|
+-----+---------+----------+
|  0.0|     7220|     37.57|
|  1.0|    11998|     62.43|
+-----+---------+----------+

Distribución de label en prueba:
+-----+---------+----------+
|label|registros|porcentaje|
+-----+---------+----------+
|  0.0|     1836|     38.39|
|  1.0|     2946|     61.61|
+-----+---------+----------+



# 6. Construcción de modelos de aprendizaje supervisado y no supervisado

En esta sección se ejecutan dos experimentos separados:

1. **Aprendizaje supervisado:** clasificación binaria para predecir si un producto fue recomprado (`reordered`).
2. **Aprendizaje no supervisado:** clustering para agrupar eventos de compra con características similares.

## 6.1 Modelo supervisado: RandomForestClassifier

La variable objetivo es `label`, derivada de `reordered`:

- `0`: producto no recomprado.
- `1`: producto recomprado.

Las variables predictoras incluyen información temporal, frecuencia de recompra, posición del producto en el carrito y categorías de producto. Las variables categóricas se procesan con `StringIndexer` y `OneHotEncoder`, mientras que las variables numéricas se integran mediante `VectorAssembler`.

In [14]:
# Variables para el modelo supervisado.
features_numericas_supervisado = [
    "order_dow",
    "order_hour_of_day",
    "days_since_prior_order",
    "add_to_cart_order"
]

features_categoricas_supervisado = [
    "department",
    "aisle",
    "bloque_horario",
    "frecuencia_recompra",
    "grupo_departamento"
]

indexers_supervisado = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in features_categoricas_supervisado
]

encoder_supervisado = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in features_categoricas_supervisado],
    outputCols=[f"{c}_ohe" for c in features_categoricas_supervisado]
)

assembler_supervisado = VectorAssembler(
    inputCols=features_numericas_supervisado + [f"{c}_ohe" for c in features_categoricas_supervisado],
    outputCol="features"
)

rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    predictionCol="prediction",
    numTrees=30,
    maxDepth=7,
    seed=seed
)

pipeline_rf = Pipeline(stages=indexers_supervisado + [encoder_supervisado, assembler_supervisado, rf])

modelo_rf = pipeline_rf.fit(train_df)
predicciones_rf = modelo_rf.transform(test_df).cache()

predicciones_rf.select(
    "label", "prediction", "probability", "department", "aisle",
    "order_hour_of_day", "days_since_prior_order", "add_to_cart_order"
).show(10, truncate=False)

26/05/31 07:15:26 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+-----+----------+----------------------------------------+-------------+------------------------+-----------------+----------------------+-----------------+
|label|prediction|probability                             |department   |aisle                   |order_hour_of_day|days_since_prior_order|add_to_cart_order|
+-----+----------+----------------------------------------+-------------+------------------------+-----------------+----------------------+-----------------+
|0.0  |1.0       |[0.4681060799189869,0.5318939200810131] |household    |paper goods             |1.0              |19.0                  |8.0              |
|0.0  |1.0       |[0.46163082714818293,0.5383691728518172]|household    |laundry                 |1.0              |19.0                  |6.0              |
|0.0  |1.0       |[0.40256347057649317,0.5974365294235068]|canned goods |canned jarred vegetables|4.0              |20.0                  |5.0              |
|0.0  |0.0       |[0.5174605924888768,0.482539407511

In [15]:
# Evaluación del modelo supervisado.
evaluador_accuracy = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
evaluador_f1 = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
)
evaluador_precision = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision"
)
evaluador_recall = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall"
)
evaluador_auc = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)

accuracy = evaluador_accuracy.evaluate(predicciones_rf)
f1 = evaluador_f1.evaluate(predicciones_rf)
precision = evaluador_precision.evaluate(predicciones_rf)
recall = evaluador_recall.evaluate(predicciones_rf)
auc = evaluador_auc.evaluate(predicciones_rf)

print("Resultados del modelo RandomForestClassifier")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1-score: {f1:.4f}")
print(f"Precision ponderada: {precision:.4f}")
print(f"Recall ponderado: {recall:.4f}")
print(f"Área bajo ROC: {auc:.4f}")

print("Matriz de confusión:")
predicciones_rf.groupBy("label", "prediction") \
    .agg(count("*").alias("registros")) \
    .orderBy("label", "prediction") \
    .show()

Resultados del modelo RandomForestClassifier
Accuracy: 0.6435
F1-score: 0.5578
Precision ponderada: 0.6587
Recall ponderado: 0.6435
Área bajo ROC: 0.6648
Matriz de confusión:
+-----+----------+---------+
|label|prediction|registros|
+-----+----------+---------+
|  0.0|       0.0|      239|
|  0.0|       1.0|     1597|
|  1.0|       0.0|      108|
|  1.0|       1.0|     2838|
+-----+----------+---------+



### Comentarios del Modelo supervisado

El modelo **RandomForestClassifier** fue entrenado para predecir si un producto dentro de una orden corresponde a una recompra (`label = 1`) o no (`label = 0`). En el conjunto de prueba se obtuvo un **accuracy de 0.6435**, un **F1-score de 0.5578**, una **precisión ponderada de 0.6587**, un **recall ponderado de 0.6435** y un **área bajo la curva ROC de 0.6648**.

Estos resultados muestran que el modelo logra una capacidad predictiva moderada. El área bajo la curva ROC mayor a 0.5 indica que el modelo sí tiene capacidad para separar parcialmente las clases, aunque todavía existe margen de mejora. La matriz de confusión muestra que el modelo identifica correctamente una parte importante de los productos recomprados, con 2,838 verdaderos positivos; sin embargo, también clasifica 1,597 productos no recomprados como recompra, lo que indica una tendencia a favorecer la clase `1`.

Este comportamiento puede explicarse porque la clase de recompra tiene mayor presencia en la muestra, aproximadamente 62% de los registros. Además, las variables utilizadas describen características generales de la orden y del producto, pero no incorporan todo el historial individual de compra de cada usuario o producto. Para mejorar el desempeño podrían agregarse variables históricas, como frecuencia previa de compra por usuario, popularidad del producto, número de compras anteriores del usuario o tasa histórica de recompra por producto.


## 6.2 Modelo no supervisado: KMeans

Para el aprendizaje no supervisado se utiliza KMeans, cuyo objetivo es agrupar eventos de compra con características similares. En este caso no se utiliza una variable objetivo. Las características consideradas incluyen variables temporales, frecuencia de recompra, posición en el carrito, recompra y categorías de producto.

Antes de aplicar KMeans, las variables categóricas se transforman con `StringIndexer` y `OneHotEncoder`, y posteriormente se aplica `StandardScaler` para escalar el vector de características.

In [16]:
features_numericas_cluster = [
    "order_dow",
    "order_hour_of_day",
    "days_since_prior_order",
    "add_to_cart_order",
    "label"
]

features_categoricas_cluster = [
    "department",
    "bloque_horario",
    "frecuencia_recompra",
    "grupo_departamento"
]

indexers_cluster = [
    StringIndexer(inputCol=c, outputCol=f"cluster_{c}_idx", handleInvalid="keep")
    for c in features_categoricas_cluster
]

encoder_cluster = OneHotEncoder(
    inputCols=[f"cluster_{c}_idx" for c in features_categoricas_cluster],
    outputCols=[f"cluster_{c}_ohe" for c in features_categoricas_cluster]
)

assembler_cluster = VectorAssembler(
    inputCols=features_numericas_cluster + [f"cluster_{c}_ohe" for c in features_categoricas_cluster],
    outputCol="features_cluster_raw"
)

scaler_cluster = StandardScaler(
    inputCol="features_cluster_raw",
    outputCol="scaled_features",
    withStd=True,
    withMean=False
)

kmeans = KMeans(
    featuresCol="scaled_features",
    predictionCol="cluster",
    k=4,
    seed=seed,
    maxIter=20
)

pipeline_kmeans = Pipeline(stages=indexers_cluster + [encoder_cluster, assembler_cluster, scaler_cluster, kmeans])

modelo_kmeans = pipeline_kmeans.fit(muestra_limpia)
predicciones_kmeans = modelo_kmeans.transform(muestra_limpia).cache()

predicciones_kmeans.select(
    "cluster", "department", "grupo_departamento", "bloque_horario",
    "frecuencia_recompra", "order_hour_of_day", "days_since_prior_order",
    "add_to_cart_order", "label"
).show(10, truncate=False)

26/05/31 07:19:28 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


+-------+-------------+------------------+--------------+-------------------+-----------------+----------------------+-----------------+-----+
|cluster|department   |grupo_departamento|bloque_horario|frecuencia_recompra|order_hour_of_day|days_since_prior_order|add_to_cart_order|label|
+-------+-------------+------------------+--------------+-------------------+-----------------+----------------------+-----------------+-----+
|2      |bakery       |otros             |madrugada 0-5 |tardía 15-30 días  |5.0              |27.0                  |1.0              |0.0  |
|2      |bakery       |otros             |madrugada 0-5 |tardía 15-30 días  |1.0              |28.0                  |10.0             |1.0  |
|2      |bakery       |otros             |madrugada 0-5 |tardía 15-30 días  |1.0              |17.0                  |9.0              |1.0  |
|1      |household    |otros             |madrugada 0-5 |tardía 15-30 días  |1.0              |19.0                  |8.0              |0.0  |

In [17]:
# Evaluación del modelo no supervisado mediante Silhouette Score.
evaluador_clustering = ClusteringEvaluator(
    featuresCol="scaled_features",
    predictionCol="cluster",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

silhouette = evaluador_clustering.evaluate(predicciones_kmeans)
print(f"Silhouette Score: {silhouette:.4f}")

print("Distribución de registros por cluster:")
predicciones_kmeans.groupBy("cluster") \
    .agg(count("*").alias("registros")) \
    .orderBy("cluster") \
    .show()

print("Perfil numérico promedio por cluster:")
predicciones_kmeans.groupBy("cluster") \
    .agg(
        count("*").alias("registros"),
        spark_round(avg("order_hour_of_day"), 2).alias("hora_promedio"),
        spark_round(avg("days_since_prior_order"), 2).alias("dias_desde_orden_previa_prom"),
        spark_round(avg("add_to_cart_order"), 2).alias("posicion_carrito_prom"),
        spark_round(avg("label"), 2).alias("proporcion_recompra")
    ) \
    .orderBy("cluster") \
    .show(truncate=False)

Silhouette Score: 0.1875
Distribución de registros por cluster:
+-------+---------+
|cluster|registros|
+-------+---------+
|      0|    12000|
|      1|     5173|
|      2|      827|
|      3|     6000|
+-------+---------+

Perfil numérico promedio por cluster:
+-------+---------+-------------+----------------------------+---------------------+-------------------+
|cluster|registros|hora_promedio|dias_desde_orden_previa_prom|posicion_carrito_prom|proporcion_recompra|
+-------+---------+-------------+----------------------------+---------------------+-------------------+
|0      |12000    |11.36        |13.89                       |8.04                 |0.69               |
|1      |5173     |11.29        |13.79                       |9.76                 |0.53               |
|2      |827      |11.54        |14.2                        |8.41                 |0.66               |
|3      |6000     |11.37        |13.93                       |9.16                 |0.57               |
+-

In [18]:
print("Distribución de grupo_departamento por cluster:")
predicciones_kmeans.groupBy("cluster", "grupo_departamento") \
    .agg(count("*").alias("registros")) \
    .orderBy("cluster", desc("registros")) \
    .show(50, truncate=False)

print("Distribución de bloque_horario por cluster:")
predicciones_kmeans.groupBy("cluster", "bloque_horario") \
    .agg(count("*").alias("registros")) \
    .orderBy("cluster", desc("registros")) \
    .show(50, truncate=False)

Distribución de grupo_departamento por cluster:
+-------+------------------+---------+
|cluster|grupo_departamento|registros|
+-------+------------------+---------+
|0      |básicos           |6000     |
|0      |frescos           |6000     |
|1      |otros             |5173     |
|2      |otros             |827      |
|3      |consumo recurrente|6000     |
+-------+------------------+---------+

Distribución de bloque_horario por cluster:
+-------+--------------+---------+
|cluster|bloque_horario|registros|
+-------+--------------+---------+
|0      |tarde 12-17   |3000     |
|0      |mañana 6-11   |3000     |
|0      |noche 18-23   |3000     |
|0      |madrugada 0-5 |3000     |
|1      |madrugada 0-5 |1312     |
|1      |noche 18-23   |1292     |
|1      |mañana 6-11   |1292     |
|1      |tarde 12-17   |1277     |
|2      |tarde 12-17   |223      |
|2      |noche 18-23   |208      |
|2      |mañana 6-11   |208      |
|2      |madrugada 0-5 |188      |
|3      |madrugada 0-5 |1500   

### Comentarios del Modelo no supervisado

El modelo **KMeans** fue aplicado para agrupar eventos de compra con características similares. Para evaluar la calidad del agrupamiento se utilizó el **Silhouette Score**, obteniendo un valor de **0.1875**. Este resultado sugiere que los grupos presentan cierta estructura, aunque no están completamente separados. En términos generales, valores cercanos a 1 representan clusters bien separados, mientras que valores cercanos a 0 indican mayor solapamiento entre grupos.

La distribución de registros por cluster fue desigual: el cluster 0 concentró 12,000 registros, el cluster 3 agrupó 6,000, el cluster 1 agrupó 5,173 y el cluster 2 únicamente 827. Al revisar el perfil de los clusters, se observa que el agrupamiento quedó fuertemente influenciado por las variables relacionadas con el tipo de producto. Por ejemplo, el cluster 0 concentra productos de los grupos `básicos` y `frescos`, el cluster 3 concentra `consumo recurrente`, mientras que los clusters 1 y 2 agrupan registros clasificados como `otros`.

En cuanto al comportamiento promedio, los clusters presentan horarios y días desde la orden previa relativamente similares, con horas promedio cercanas a 11 y días desde la orden previa entre 13.79 y 14.20. Esto indica que, para esta muestra, la separación principal no se dio tanto por variables temporales, sino por la categoría del producto. Aun así, el experimento permite observar cómo PySpark puede utilizarse para segmentar registros de compra y explorar patrones no etiquetados dentro de grandes volúmenes de datos.

# 7. Conclusiones

En esta actividad se construyó una muestra representativa **M** a partir del dataset Instacart Market Basket Analysis, utilizando un muestreo estratificado por particiones derivadas del horario de compra, la frecuencia de recompra y el grupo de departamento. La muestra final quedó conformada por **24,000 registros**, tomando **500 registros por cada una de las 48 particiones**, lo que permitió mantener diversidad en los datos y evitar que las particiones con mayor volumen dominaran completamente el análisis.

Posteriormente, se realizó la preparación de datos mediante selección de variables, validación de valores nulos, conversión de tipos de datos, creación de variables categóricas e integración de columnas en vectores de características. La muestra fue dividida en conjuntos de entrenamiento y prueba mediante una estrategia estratificada, conservando una distribución similar de la variable objetivo: en entrenamiento, la clase de recompra representó 62.43%, mientras que en prueba representó 61.61%.

En el experimento supervisado, el modelo **RandomForestClassifier** obtuvo un accuracy de **0.6435** y un área bajo ROC de **0.6648**, lo que refleja un desempeño moderado para predecir si un producto fue recomprado. El modelo logró identificar una parte importante de los productos recomprados, aunque mostró tendencia a clasificar registros como recompra, generando falsos positivos.

En el experimento no supervisado, el algoritmo **KMeans** obtuvo un **Silhouette Score de 0.1875**. Esto indica que los clusters presentan cierta estructura, aunque con separación limitada. La interpretación de los grupos mostró que el agrupamiento estuvo influenciado principalmente por el tipo de producto, especialmente por los grupos de departamento.

En conjunto, los resultados muestran cómo PySpark permite implementar un flujo completo de aprendizaje automático sobre datos de gran volumen, desde la construcción de una muestra representativa hasta la preparación de datos, división entrenamiento-prueba, entrenamiento de modelos y evaluación de resultados. Además, la actividad permitió identificar posibles mejoras futuras, como incorporar variables históricas a nivel usuario-producto o probar diferentes configuraciones de modelos para mejorar la capacidad predictiva y la calidad de los clusters.

# 8. Referencias

Apache Spark. (2026). *Classification and regression*. Spark MLlib. https://spark.apache.org/docs/latest/ml-classification-regression.html

Apache Spark. (2026). *Clustering*. Spark MLlib. https://spark.apache.org/docs/latest/ml-clustering.html

Instacart. (2017). *Instacart Market Basket Analysis*. Kaggle. https://www.kaggle.com/datasets/psparks/instacart-market-basket-analysis

Polak, A. (2023). *Scaling machine learning with Spark: Distributed ML with MLlib, TensorFlow, and PyTorch*. O'Reilly Media.